# 01 — Data Cleaning & Validation

Wejście: `data/raw/Hotels.csv` · Wyjście: `data/processed/hotels_clean.parquet` + `data/sample/hotels_sample.csv`

**TODO:**
- konwersja serial Excela -> daty (Checkin/Checkout)
- obsługa ujemnych `Net_Revenue` / `GST` (anulacje)
- typy, sanity checks, walidacja
- zapis czystego pliku

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from pathlib import Path

In [8]:
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Hotels.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()
# Checkin and checkout are intigers, but they should be datetime


,Booking_ID,Hotel_Name,Hotel_Category,City,State,Country,Checkin_Date,Checkout_Date,Month,Quarter,...,Net_Revenue,GST_Amount,Total_With_GST,Booking_Channel,Customer_Type,Payment_Mode,Occupancy_Status,Customer_Rating,Feedback_Status,Covid_Impact_Level
0,BK9835534,ITC Kakatiya,Luxury,Hyderabad,Telangana,India,44320,44324,May,Q2,...,89160,10699.20,99859.20,Walk-in,Regular,UPI,Checked-Out,4.6,Positive,High
1,BK7803377,ITC Royal Bengal,Luxury,Kolkata,West Bengal,India,44352,44356,June,Q2,...,86830,10419.60,97249.60,Corporate,VIP,Credit Card,Checked-In,4.6,Positive,High
2,BK3686278,ITC Grand Goa,Resort,Goa,Goa,India,44443,44445,September,Q3,...,29389,3526.68,32915.68,Booking.com,Corporate,Cash,Checked-In,4.2,Positive,Low
3,BK7514886,Fortune Select Trinity,Business,Bengaluru,Karnataka,India,44469,44470,September,Q3,...,24409,2929.08,27338.08,Website,Regular,Debit Card,Cancelled,3.8,Neutral,Low
4,BK6312815,ITC Narmada,Business,Ahmedabad,Gujarat,India,44461,44463,September,Q3,...,9940,1192.80,11132.80,Walk-in,New Guest,UPI,Checked-In,4.6,Negative,Low


In [ ]:
df.info()
df.describe()
df.isnull().sum()

In [ ]:
df['Checkin_Date_time'] = pd.to_datetime(df['Checkin_Date'], unit='D', origin='1899-12-30')
df['Checkout_Date_time'] = pd.to_datetime(df['Checkout_Date'], unit='D', origin='1899-12-30')
df.head()

,Booking_ID,Hotel_Name,Hotel_Category,City,State,Country,Checkin_Date,Checkout_Date,Month,Quarter,...,Total_With_GST,Booking_Channel,Customer_Type,Payment_Mode,Occupancy_Status,Customer_Rating,Feedback_Status,Covid_Impact_Level,Checkin_Date_time,Checkout_Date_time
0,BK9835534,ITC Kakatiya,Luxury,Hyderabad,Telangana,India,44320,44324,May,Q2,...,99859.20,Walk-in,Regular,UPI,Checked-Out,4.6,Positive,High,2021-05-04,2021-05-08
1,BK7803377,ITC Royal Bengal,Luxury,Kolkata,West Bengal,India,44352,44356,June,Q2,...,97249.60,Corporate,VIP,Credit Card,Checked-In,4.6,Positive,High,2021-06-05,2021-06-09
2,BK3686278,ITC Grand Goa,Resort,Goa,Goa,India,44443,44445,September,Q3,...,32915.68,Booking.com,Corporate,Cash,Checked-In,4.2,Positive,Low,2021-09-04,2021-09-06
3,BK7514886,Fortune Select Trinity,Business,Bengaluru,Karnataka,India,44469,44470,September,Q3,...,27338.08,Website,Regular,Debit Card,Cancelled,3.8,Neutral,Low,2021-09-30,2021-10-01
4,BK6312815,ITC Narmada,Business,Ahmedabad,Gujarat,India,44461,44463,September,Q3,...,11132.80,Walk-in,New Guest,UPI,Checked-In,4.6,Negative,Low,2021-09-22,2021-09-24


In [ ]:
print(df['Checkin_Date_time'].max(), df['Checkin_Date_time'].min())
print(df['Checkout_Date_time'].max(), df['Checkout_Date_time'].min())
# Checkin and checkout dates are between 2021 and 2022, which is good.

2022-03-31 00:00:00 2021-04-01 00:00:00
2022-04-05 00:00:00 2021-04-02 00:00:00


In [29]:
df['nights_calc'] = (df['Checkout_Date_time'] - df['Checkin_Date_time']).dt.days
print(df.head())
print('Checkout < Checkin:', (df['Checkout_Date_time'] < df['Checkin_Date_time']).sum())
print('nights_calc != Nights_Stayed:', (df['nights_calc'] != df['Nights_Stayed']).sum())
# Nights_Stayed and nights_calc are the same, which is good. There are no cases where checkout is before checkin, which is also good.

  Booking_ID              Hotel_Name Hotel_Category       City        State  \
0  BK9835534            ITC Kakatiya         Luxury  Hyderabad    Telangana   
1  BK7803377        ITC Royal Bengal         Luxury    Kolkata  West Bengal   
2  BK3686278           ITC Grand Goa         Resort        Goa          Goa   
3  BK7514886  Fortune Select Trinity       Business  Bengaluru    Karnataka   
4  BK6312815             ITC Narmada       Business  Ahmedabad      Gujarat   

  Country  Checkin_Date  Checkout_Date      Month Quarter  ...  \
0   India         44320          44324        May      Q2  ...   
1   India         44352          44356       June      Q2  ...   
2   India         44443          44445  September      Q3  ...   
3   India         44469          44470  September      Q3  ...   
4   India         44461          44463  September      Q3  ...   

   Booking_Channel Customer_Type  Payment_Mode  Occupancy_Status  \
0          Walk-in       Regular           UPI       Checked

In [ ]:
print('Year incorrect:', (df['Year'] != df['Checkin_Date_time'].dt.year).sum())
print('Month incorrect:',(df['Month'] != df['Checkin_Date_time'].dt.strftime('%B')).sum())
# Maintaining data consistency and accurate documentation 

Year incorrect: 0
Month incorrect: 0
